# Q5 Training EN

This notebook runs the full **Question 5** pipeline for **English** only.

It does all of the following in one place:
- trains the English byte-level BPE tokenizer
- tokenizes the English corpus into BPE subword pieces
- trains the English subword NNLM
- saves a resumable checkpoint every 5 epochs to `checkpoint_bpe_nnlm_en.pt`
- resumes automatically if that checkpoint already exists
- writes the exact autograder files for this language
- writes the Question 5 comparison tables and draft answers for this language
- packages everything into `q5_en_artifacts.zip`

This notebook is a standalone Question 5 notebook.
It does **not** require running the Question 3 or Question 4 notebooks inside the same session first.


## Install Packages

In [4]:
import subprocess
import sys

packages = [
    "datasets",
    "matplotlib",
    "nltk",
]

subprocess.run([sys.executable, "-m", "pip", "install", "-q", *packages], check=True)


CompletedProcess(args=['/usr/bin/python3', '-m', 'pip', 'install', '-q', 'datasets', 'matplotlib', 'nltk'], returncode=0)

## Bootstrap Shared Runtime

This cell recreates `q5_parallel_runtime.py` automatically inside Kaggle,
so the notebook works even if you uploaded only the `.ipynb` file.


In [5]:
from pathlib import Path

runtime_path = Path("q5_parallel_runtime.py")
runtime_path.write_text('from __future__ import annotations\n\nimport copy\nimport heapq\nimport json\nimport math\nimport os\nimport pickle\nimport random\nimport re\nimport shutil\nimport time\nimport zipfile\nfrom array import array\nfrom collections import Counter, defaultdict\nfrom getpass import getpass\nfrom pathlib import Path\n\nimport matplotlib.pyplot as plt\nimport nltk\nimport numpy as np\nimport pandas as pd\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\nimport torch.optim as optim\nfrom datasets import load_dataset\n\n\nnltk.download("punkt", quiet=True)\ntry:\n    nltk.download("punkt_tab", quiet=True)\nexcept Exception:\n    pass\n\n\nLANGS = ("en", "hi")\nSPECIALS = ["<pad>", "<unk>", "<s>", "</s>"]\nSPECIAL_SET = set(SPECIALS)\n\nSHARED_SEED = 42\nMIN_TOKENS = 3\n\nEN_CACHE = "shared_english_corpus.csv"\nHI_CACHE = "shared_hindi_corpus.csv"\n\nEMBED_DIM = 100\nDEFAULT_Q5_N = 4\nDEFAULT_Q5_HIDDEN = 100\nDEFAULT_Q5_EPOCHS = 30\nDEFAULT_Q5_BATCH_SIZE = 512\nDEFAULT_Q5_LR = 1e-3\nDEFAULT_Q5_WEIGHT_DECAY = 1e-4\nDEFAULT_Q5_PATIENCE = 3\nDEFAULT_DIRECT = False\nDEFAULT_TARGET_VOCAB_SIZE = 5000\nCHECKPOINT_EVERY = 5\n\nTOKENIZER_SUFFIX = "</w>"\nMERGE_PROGRESS_EVERY = 100\nTOKENIZE_PROGRESS_EVERY = 50000\n\n\ndef build_default_config():\n    return {\n        "n": DEFAULT_Q5_N,\n        "m": EMBED_DIM,\n        "h": DEFAULT_Q5_HIDDEN,\n        "epochs": DEFAULT_Q5_EPOCHS,\n        "batch_size": DEFAULT_Q5_BATCH_SIZE,\n        "lr": DEFAULT_Q5_LR,\n        "weight_decay": DEFAULT_Q5_WEIGHT_DECAY,\n        "patience": DEFAULT_Q5_PATIENCE,\n        "direct": DEFAULT_DIRECT,\n        "target_vocab_size": DEFAULT_TARGET_VOCAB_SIZE,\n        "max_pairs": None,\n        "hf_token": os.environ.get("HF_TOKEN", "").strip(),\n    }\n\n\ndef language_label(lang: str) -> str:\n    return "English" if lang == "en" else "Hindi"\n\n\ndef cache_path_for_lang(lang: str) -> Path:\n    return Path(EN_CACHE if lang == "en" else HI_CACHE)\n\n\ndef q5_files(lang: str):\n    return {\n        "tokenizer": Path(f"bpe_{lang}.json"),\n        "model": Path(f"bpe_nnlm_{lang}.pt"),\n        "checkpoint": Path(f"checkpoint_bpe_nnlm_{lang}.pt"),\n        "train_log": Path(f"bpe_train_log_{lang}.json"),\n        "train_state": Path(f"q5_train_state_{lang}.pkl"),\n        "train_summary": Path(f"q5_train_summary_{lang}.json"),\n        "bpe_summary": Path(f"q5_bpe_summary_{lang}.json"),\n        "plot": Path(f"q5_convergence_{lang}.png"),\n        "perplexity_compare": Path(f"q5_perplexity_compare_{lang}.csv"),\n        "parameter_compare": Path(f"q5_parameter_compare_{lang}.csv"),\n        "embedding_compare": Path(f"q5_embedding_compare_{lang}.csv"),\n        "analysis_summary": Path(f"q5_analysis_summary_{lang}.json"),\n        "written_answers": Path(f"Q5_WRITTEN_ANSWERS_{lang}.md"),\n        "bundle": Path(f"q5_{lang}_artifacts.zip"),\n    }\n\n\ndef set_shared_seed(seed: int = SHARED_SEED):\n    random.seed(seed)\n    np.random.seed(seed)\n    torch.manual_seed(seed)\n    if torch.cuda.is_available():\n        torch.cuda.manual_seed_all(seed)\n    return seed\n\n\ndef format_duration(seconds: float) -> str:\n    seconds = max(0, int(seconds))\n    hours, rem = divmod(seconds, 3600)\n    minutes, secs = divmod(rem, 60)\n    return f"{hours:02d}:{minutes:02d}:{secs:02d}"\n\n\ndef shared_tokenize(text):\n    return nltk.word_tokenize(str(text).lower())\n\n\ndef surface_tokenize(text):\n    return nltk.word_tokenize(str(text))\n\n\ndef get_hf_token(config: dict | None = None) -> str:\n    env_token = os.environ.get("HF_TOKEN", "").strip()\n    if env_token:\n        return env_token\n\n    if config and config.get("hf_token"):\n        token = str(config["hf_token"]).strip()\n        if token:\n            os.environ["HF_TOKEN"] = token\n            return token\n\n    token = getpass("Enter Hugging Face token: ").strip()\n    if not token:\n        raise ValueError("A Hugging Face token is required to download the assignment dataset.")\n    os.environ["HF_TOKEN"] = token\n    return token\n\n\ndef build_shared_corpus_files(seed: int = SHARED_SEED, config: dict | None = None):\n    ds = load_dataset(\n        "Exploration-Lab/CS781",\n        data_files={"train": "Assignment-1/train.parquet"},\n        split="train",\n        token=get_hf_token(config),\n    )\n\n    translations = ds["translation"]\n    en_df = pd.DataFrame({"text": [row["en"] for row in translations]})\n    hi_df = pd.DataFrame({"text": [row["hi"] for row in translations]})\n\n    en_df = en_df.dropna(subset=["text"]).drop_duplicates(subset=["text"])\n    hi_df = hi_df.dropna(subset=["text"]).drop_duplicates(subset=["text"])\n\n    en_df = en_df[en_df["text"].apply(lambda x: len(shared_tokenize(x)) >= MIN_TOKENS)]\n    hi_df = hi_df[hi_df["text"].apply(lambda x: len(shared_tokenize(x)) >= MIN_TOKENS)]\n\n    en_order = np.random.RandomState(seed).permutation(len(en_df))\n    hi_order = np.random.RandomState(seed).permutation(len(hi_df))\n    en_df = en_df.iloc[en_order].reset_index(drop=True)\n    hi_df = hi_df.iloc[hi_order].reset_index(drop=True)\n\n    en_df.to_csv(EN_CACHE, index=False)\n    hi_df.to_csv(HI_CACHE, index=False)\n    print(f"Wrote {len(en_df):,} English sentences to {EN_CACHE}")\n    print(f"Wrote {len(hi_df):,} Hindi sentences to {HI_CACHE}")\n\n\ndef load_language_texts(lang: str, config: dict) -> list[str]:\n    path = cache_path_for_lang(lang)\n    if not path.exists():\n        build_shared_corpus_files(config=config)\n    df = pd.read_csv(path, keep_default_na=False)\n    texts = df["text"].tolist()\n    max_pairs = config.get("max_pairs")\n    return texts[:max_pairs] if max_pairs is not None else texts\n\n\ndef split_texts(texts, ratios=(0.8, 0.1, 0.1)):\n    total = len(texts)\n    train_end = int(total * ratios[0])\n    valid_end = train_end + int(total * ratios[1])\n    train = texts[:train_end]\n    valid = texts[train_end:valid_end]\n    test = texts[valid_end:]\n    return train, valid, test\n\n\ndef make_symbol(byte_values, terminal):\n    return (tuple(int(b) for b in byte_values), bool(terminal))\n\n\ndef symbol_bytes(symbol):\n    return symbol[0]\n\n\ndef symbol_terminal(symbol):\n    return bool(symbol[1])\n\n\ndef merge_symbols(left, right):\n    return make_symbol(symbol_bytes(left) + symbol_bytes(right), symbol_terminal(right))\n\n\ndef initial_symbols_for_word(word: str):\n    payload = str(word).encode("utf-8")\n    if not payload:\n        return []\n    if len(payload) == 1:\n        return [make_symbol((payload[0],), True)]\n    seq = [make_symbol((value,), False) for value in payload[:-1]]\n    seq.append(make_symbol((payload[-1],), True))\n    return seq\n\n\ndef sequence_pair_counts(symbols):\n    counts = Counter()\n    for idx in range(len(symbols) - 1):\n        counts[(symbols[idx], symbols[idx + 1])] += 1\n    return counts\n\n\ndef merge_pair_in_sequence(symbols, pair, merged_symbol):\n    out = []\n    idx = 0\n    left, right = pair\n    while idx < len(symbols):\n        if idx + 1 < len(symbols) and symbols[idx] == left and symbols[idx + 1] == right:\n            out.append(merged_symbol)\n            idx += 2\n        else:\n            out.append(symbols[idx])\n            idx += 1\n    return out\n\n\ndef bytes_to_display(byte_values):\n    payload = bytes(byte_values)\n    try:\n        text = payload.decode("utf-8")\n        if text and TOKENIZER_SUFFIX not in text and all(ch not in "\\x00\\r\\n\\t" for ch in text):\n            return text\n    except UnicodeDecodeError:\n        pass\n    return "".join(f"<0x{value:02X}>" for value in payload)\n\n\ndef symbol_to_token(symbol):\n    body = bytes_to_display(symbol_bytes(symbol))\n    return body + TOKENIZER_SUFFIX if symbol_terminal(symbol) else body\n\n\ndef token_to_symbol(token: str):\n    token = str(token)\n    terminal = token.endswith(TOKENIZER_SUFFIX)\n    body = token[: -len(TOKENIZER_SUFFIX)] if terminal else token\n    payload = bytearray()\n    idx = 0\n    while idx < len(body):\n        match = re.match(r"<0x([0-9A-Fa-f]{2})>", body[idx:])\n        if match:\n            payload.append(int(match.group(1), 16))\n            idx += len(match.group(0))\n            continue\n        payload.extend(body[idx].encode("utf-8"))\n        idx += 1\n    return make_symbol(tuple(payload), terminal)\n\n\ndef token_to_surface_word(token: str):\n    symbol = token_to_symbol(token)\n    if not symbol_terminal(symbol):\n        return None\n    try:\n        return bytes(symbol_bytes(symbol)).decode("utf-8")\n    except UnicodeDecodeError:\n        return None\n\n\nclass ByteLevelBPE:\n    def __init__(self, target_vocab_size=DEFAULT_TARGET_VOCAB_SIZE, lang: str | None = None):\n        self.target_vocab_size = int(target_vocab_size)\n        self.lang = lang\n        self.base_symbols = []\n        self.merged_symbols = []\n        self.merge_rules = []\n        self.merge_lookup = {}\n        self.merge_rank = {}\n        self.itos = list(SPECIALS)\n        self.stoi = {tok: idx for idx, tok in enumerate(self.itos)}\n        self.word_cache = {}\n        self.lowercase_representatives = {}\n        self.summary = {}\n\n    @property\n    def vocab_size(self):\n        return len(self.itos)\n\n    def fit(self, texts, verbose=True):\n        word_counter = Counter()\n        word_first_seen = {}\n        lower_variant_counts = defaultdict(Counter)\n        next_word_order = 0\n\n        base_symbols = []\n        base_seen = set()\n        total_surface_tokens = 0\n\n        started = time.time()\n        for text_idx, text in enumerate(texts, start=1):\n            tokens = surface_tokenize(text)\n            total_surface_tokens += len(tokens)\n            for token in tokens:\n                word_counter[token] += 1\n                lower_variant_counts[token.lower()][token] += 1\n                if token not in word_first_seen:\n                    word_first_seen[token] = next_word_order\n                    next_word_order += 1\n                    for symbol in initial_symbols_for_word(token):\n                        if symbol not in base_seen:\n                            base_seen.add(symbol)\n                            base_symbols.append(symbol)\n            if verbose and text_idx % TOKENIZE_PROGRESS_EVERY == 0:\n                print(\n                    f"[BPE setup] {language_label(self.lang or \'en\')} | "\n                    f"sentences {text_idx:,}/{len(texts):,} | "\n                    f"surface tokens {total_surface_tokens:,} | "\n                    f"unique surface tokens {len(word_counter):,} | "\n                    f"elapsed {format_duration(time.time() - started)}",\n                    flush=True,\n                )\n\n        words = sorted(word_counter, key=word_first_seen.get)\n        word_freqs = [int(word_counter[word]) for word in words]\n        word_sequences = [initial_symbols_for_word(word) for word in words]\n\n        pair_freqs = Counter()\n        pair_to_words = defaultdict(set)\n        for word_idx, seq in enumerate(word_sequences):\n            local_counts = sequence_pair_counts(seq)\n            if not local_counts:\n                continue\n            weight = word_freqs[word_idx]\n            for pair, count in local_counts.items():\n                pair_freqs[pair] += weight * count\n                pair_to_words[pair].add(word_idx)\n\n        heap = [(-int(freq), pair) for pair, freq in pair_freqs.items()]\n        heapq.heapify(heap)\n\n        merged_symbols = []\n        merged_seen = set()\n        merge_rules = []\n        merge_steps = 0\n        target = max(self.target_vocab_size, len(SPECIALS) + len(base_symbols))\n\n        while len(SPECIALS) + len(base_symbols) + len(merged_symbols) < target and heap:\n            neg_freq, pair = heapq.heappop(heap)\n            current_freq = int(pair_freqs.get(pair, 0))\n            if current_freq != -neg_freq:\n                continue\n            if current_freq <= 1:\n                break\n\n            merged_symbol = merge_symbols(pair[0], pair[1])\n            merge_rules.append((pair[0], pair[1], merged_symbol))\n            merge_steps += 1\n\n            if merged_symbol not in base_seen and merged_symbol not in merged_seen:\n                merged_seen.add(merged_symbol)\n                merged_symbols.append(merged_symbol)\n\n            affected = list(pair_to_words.get(pair, ()))\n            for word_idx in affected:\n                old_seq = word_sequences[word_idx]\n                old_counts = sequence_pair_counts(old_seq)\n                if pair not in old_counts:\n                    bucket = pair_to_words.get(pair)\n                    if bucket is not None:\n                        bucket.discard(word_idx)\n                        if not bucket:\n                            pair_to_words.pop(pair, None)\n                    continue\n\n                weight = word_freqs[word_idx]\n                for old_pair, count in old_counts.items():\n                    updated = int(pair_freqs.get(old_pair, 0) - weight * count)\n                    if updated > 0:\n                        pair_freqs[old_pair] = updated\n                        heapq.heappush(heap, (-updated, old_pair))\n                    else:\n                        pair_freqs.pop(old_pair, None)\n\n                    bucket = pair_to_words.get(old_pair)\n                    if bucket is not None:\n                        bucket.discard(word_idx)\n                        if not bucket:\n                            pair_to_words.pop(old_pair, None)\n\n                new_seq = merge_pair_in_sequence(old_seq, pair, merged_symbol)\n                word_sequences[word_idx] = new_seq\n                new_counts = sequence_pair_counts(new_seq)\n                for new_pair, count in new_counts.items():\n                    updated = int(pair_freqs.get(new_pair, 0) + weight * count)\n                    pair_freqs[new_pair] = updated\n                    pair_to_words[new_pair].add(word_idx)\n                    heapq.heappush(heap, (-updated, new_pair))\n\n            if verbose and (merge_steps == 1 or merge_steps % MERGE_PROGRESS_EVERY == 0):\n                effective_vocab = len(SPECIALS) + len(base_symbols) + len(merged_symbols)\n                print(\n                    f"[BPE merge] {language_label(self.lang or \'en\')} | "\n                    f"merge {merge_steps:,} | vocab {effective_vocab:,}/{target:,} | "\n                    f"best pair freq {current_freq:,} | elapsed {format_duration(time.time() - started)}",\n                    flush=True,\n                )\n\n        merge_lookup = {}\n        merge_rank = {}\n        for rank, (left, right, merged) in enumerate(merge_rules):\n            merge_lookup[(left, right)] = merged\n            merge_rank[(left, right)] = rank\n\n        representatives = {}\n        for lower, variants in lower_variant_counts.items():\n            if lower in variants:\n                representatives[lower] = lower\n                continue\n            representative = sorted(\n                variants.items(),\n                key=lambda item: (-item[1], word_first_seen[item[0]], item[0]),\n            )[0][0]\n            representatives[lower] = representative\n\n        self.base_symbols = base_symbols\n        self.merged_symbols = merged_symbols\n        self.merge_rules = merge_rules\n        self.merge_lookup = merge_lookup\n        self.merge_rank = merge_rank\n        self.lowercase_representatives = representatives\n        self.itos = (\n            list(SPECIALS)\n            + [symbol_to_token(symbol) for symbol in base_symbols]\n            + [symbol_to_token(symbol) for symbol in merged_symbols]\n        )\n        self.stoi = {token: idx for idx, token in enumerate(self.itos)}\n        self.word_cache = {}\n        self.summary = {\n            "language": language_label(self.lang or "en"),\n            "lang": self.lang,\n            "target_vocab_size": self.target_vocab_size,\n            "effective_vocab_size": len(self.itos),\n            "initial_symbol_count": len(base_symbols),\n            "learned_merges": len(merge_rules),\n            "unique_surface_tokens_train": len(words),\n            "surface_tokens_train": total_surface_tokens,\n            "elapsed_seconds": time.time() - started,\n        }\n        return self\n\n    def encode_word(self, word: str):\n        word = str(word)\n        cached = self.word_cache.get(word)\n        if cached is not None:\n            return list(cached)\n\n        symbols = initial_symbols_for_word(word)\n        if not symbols:\n            return []\n\n        while len(symbols) > 1:\n            best_rank = None\n            best_pos = None\n            best_symbol = None\n            for idx in range(len(symbols) - 1):\n                pair = (symbols[idx], symbols[idx + 1])\n                rank = self.merge_rank.get(pair)\n                if rank is None:\n                    continue\n                if best_rank is None or rank < best_rank:\n                    best_rank = rank\n                    best_pos = idx\n                    best_symbol = self.merge_lookup[pair]\n            if best_pos is None:\n                break\n            symbols = symbols[:best_pos] + [best_symbol] + symbols[best_pos + 2 :]\n\n        tokens = tuple(symbol_to_token(symbol) for symbol in symbols)\n        self.word_cache[word] = tokens\n        return list(tokens)\n\n    def tokenize_text(self, text):\n        pieces = []\n        for token in surface_tokenize(text):\n            pieces.extend(self.encode_word(token))\n        return pieces\n\n    def to_export_dict(self):\n        return {\n            "language": self.lang,\n            "target_vocab_size": self.target_vocab_size,\n            "vocab_size": len(self.itos),\n            "itos": self.itos,\n            "base_tokens": [symbol_to_token(symbol) for symbol in self.base_symbols],\n            "merge_tokens": [symbol_to_token(symbol) for symbol in self.merged_symbols],\n            "merges": [\n                {\n                    "left": symbol_to_token(left),\n                    "right": symbol_to_token(right),\n                    "merged": symbol_to_token(merged),\n                }\n                for left, right, merged in self.merge_rules\n            ],\n            "byte_level": True,\n            "case_sensitive": True,\n            "terminal_suffix": TOKENIZER_SUFFIX,\n            "summary": self.summary,\n        }\n\n    def save(self, path):\n        Path(path).write_text(\n            json.dumps(self.to_export_dict(), indent=2, ensure_ascii=False),\n            encoding="utf-8",\n        )\n\n    @classmethod\n    def load(cls, path):\n        data = json.loads(Path(path).read_text(encoding="utf-8"))\n        obj = cls(target_vocab_size=int(data.get("target_vocab_size", data["vocab_size"])), lang=data.get("language"))\n        obj.itos = list(data["itos"])\n        obj.stoi = {token: idx for idx, token in enumerate(obj.itos)}\n        obj.base_symbols = [token_to_symbol(token) for token in data.get("base_tokens", [])]\n        merged_tokens = data.get("merge_tokens", [])\n        obj.merged_symbols = [token_to_symbol(token) for token in merged_tokens]\n\n        merge_rules = []\n        for item in data.get("merges", []):\n            if isinstance(item, dict):\n                left = token_to_symbol(item["left"])\n                right = token_to_symbol(item["right"])\n                merged = token_to_symbol(item["merged"])\n            elif len(item) == 3:\n                left = token_to_symbol(item[0])\n                right = token_to_symbol(item[1])\n                merged = token_to_symbol(item[2])\n            else:\n                raise ValueError("Unsupported BPE merge format.")\n            merge_rules.append((left, right, merged))\n\n        obj.merge_rules = merge_rules\n        obj.merge_lookup = {}\n        obj.merge_rank = {}\n        for rank, (left, right, merged) in enumerate(merge_rules):\n            obj.merge_lookup[(left, right)] = merged\n            obj.merge_rank[(left, right)] = rank\n\n        obj.summary = data.get("summary", {})\n        obj.word_cache = {}\n        return obj\n\n\ndef load_or_train_bpe(lang: str, train_texts, config: dict):\n    files = q5_files(lang)\n    tokenizer_path = files["tokenizer"]\n    if tokenizer_path.exists():\n        tokenizer = ByteLevelBPE.load(tokenizer_path)\n        saved_target = int(tokenizer.summary.get("target_vocab_size", tokenizer.target_vocab_size))\n        if saved_target != int(config["target_vocab_size"]):\n            raise ValueError(\n                f"Existing {tokenizer_path.name} was trained with target_vocab_size={saved_target}, "\n                f"but current config wants {config[\'target_vocab_size\']}."\n            )\n        print(f"Loaded existing tokenizer from {tokenizer_path.name}")\n        save_json(files["bpe_summary"], tokenizer.summary or {"target_vocab_size": tokenizer.target_vocab_size})\n        return tokenizer\n\n    tokenizer = ByteLevelBPE(target_vocab_size=config["target_vocab_size"], lang=lang)\n    tokenizer.fit(train_texts, verbose=True)\n    tokenizer.save(tokenizer_path)\n    save_json(files["bpe_summary"], tokenizer.summary)\n    print(f"Saved tokenizer to {tokenizer_path.name}")\n    return tokenizer\n\n\ndef build_surface_stats(texts):\n    lower_counts = Counter()\n    lower_variant_counts = defaultdict(Counter)\n    first_seen = {}\n    order = 0\n    total_surface_tokens = 0\n\n    for text in texts:\n        for token in surface_tokenize(text):\n            total_surface_tokens += 1\n            lower = token.lower()\n            lower_counts[lower] += 1\n            lower_variant_counts[lower][token] += 1\n            if token not in first_seen:\n                first_seen[token] = order\n                order += 1\n\n    representatives = {}\n    for lower, variants in lower_variant_counts.items():\n        if lower in variants:\n            representatives[lower] = lower\n            continue\n        representative = sorted(\n            variants.items(),\n            key=lambda item: (-item[1], first_seen[item[0]], item[0]),\n        )[0][0]\n        representatives[lower] = representative\n\n    return {\n        "lower_counts": lower_counts,\n        "representatives": representatives,\n        "total_surface_tokens": total_surface_tokens,\n        "surface_vocab_size": len(lower_counts),\n    }\n\n\ndef vectorize_bpe_texts(texts, tokenizer: ByteLevelBPE, stoi, n: int, label: str):\n    bos = [stoi["<s>"]] * (n - 1)\n    eos = stoi["</s>"]\n\n    xbuf = array("i")\n    ybuf = array("i")\n    surface_token_total = 0\n    sentence_total = 0\n    subword_token_total = 0\n    single_piece_token_total = 0\n\n    started = time.time()\n    for text_idx, text in enumerate(texts, start=1):\n        surface_tokens = surface_tokenize(text)\n        if not surface_tokens:\n            continue\n\n        sentence_total += 1\n        surface_token_total += len(surface_tokens)\n        sent_ids = list(bos)\n        for token in surface_tokens:\n            pieces = tokenizer.encode_word(token)\n            subword_token_total += len(pieces)\n            if len(pieces) == 1:\n                single_piece_token_total += 1\n            sent_ids.extend(stoi[piece] for piece in pieces)\n        sent_ids.append(eos)\n\n        for end in range(n - 1, len(sent_ids)):\n            xbuf.extend(sent_ids[end - (n - 1) : end])\n            ybuf.append(sent_ids[end])\n\n        if text_idx % TOKENIZE_PROGRESS_EVERY == 0:\n            print(\n                f"[vectorize] {label} | sentences {text_idx:,}/{len(texts):,} | "\n                f"surface tokens {surface_token_total:,} | subword tokens {subword_token_total:,} | "\n                f"elapsed {format_duration(time.time() - started)}",\n                flush=True,\n            )\n\n    if len(ybuf) == 0:\n        X = np.empty((0, n - 1), dtype=np.int32)\n        Y = np.empty((0,), dtype=np.int32)\n    else:\n        X = np.frombuffer(xbuf, dtype=np.int32).copy().reshape(-1, n - 1)\n        Y = np.frombuffer(ybuf, dtype=np.int32).copy()\n\n    word_units = surface_token_total + sentence_total\n    stats = {\n        "sentences": sentence_total,\n        "surface_tokens": surface_token_total,\n        "surface_word_units": word_units,\n        "subword_tokens": subword_token_total,\n        "subword_targets": int(len(Y)),\n        "avg_subwords_per_surface_token": (\n            subword_token_total / surface_token_total if surface_token_total else 0.0\n        ),\n        "single_piece_rate": (\n            single_piece_token_total / surface_token_total if surface_token_total else 0.0\n        ),\n    }\n    return X, Y, stats\n\n\ndef prepare_language_resources(lang: str, config: dict):\n    texts = load_language_texts(lang, config)\n    train_texts, valid_texts, test_texts = split_texts(texts)\n\n    tokenizer = load_or_train_bpe(lang, train_texts, config)\n    stoi = tokenizer.stoi\n    itos = tokenizer.itos\n    surface_stats = build_surface_stats(train_texts)\n\n    Xtr, Ytr, train_stats = vectorize_bpe_texts(\n        train_texts,\n        tokenizer=tokenizer,\n        stoi=stoi,\n        n=config["n"],\n        label=f"{language_label(lang)} train",\n    )\n    Xva, Yva, valid_stats = vectorize_bpe_texts(\n        valid_texts,\n        tokenizer=tokenizer,\n        stoi=stoi,\n        n=config["n"],\n        label=f"{language_label(lang)} valid",\n    )\n    Xte, Yte, test_stats = vectorize_bpe_texts(\n        test_texts,\n        tokenizer=tokenizer,\n        stoi=stoi,\n        n=config["n"],\n        label=f"{language_label(lang)} test",\n    )\n\n    return {\n        "texts": texts,\n        "train_texts": train_texts,\n        "valid_texts": valid_texts,\n        "test_texts": test_texts,\n        "tokenizer": tokenizer,\n        "stoi": stoi,\n        "itos": itos,\n        "Xtr": Xtr,\n        "Ytr": Ytr,\n        "Xva": Xva,\n        "Yva": Yva,\n        "Xte": Xte,\n        "Yte": Yte,\n        "train_stats": train_stats,\n        "valid_stats": valid_stats,\n        "test_stats": test_stats,\n        "surface_stats": surface_stats,\n    }\n\n\ndef resolve_device(device_id: int | None):\n    if device_id is not None and device_id >= 0 and torch.cuda.is_available():\n        usable = min(device_id, torch.cuda.device_count() - 1)\n        torch.cuda.set_device(usable)\n        device = torch.device(f"cuda:{usable}")\n        print(f"Using {device} -> {torch.cuda.get_device_name(usable)}")\n        return device\n\n    if torch.cuda.is_available():\n        torch.cuda.set_device(0)\n        device = torch.device("cuda:0")\n        print(f"Using {device} -> {torch.cuda.get_device_name(0)}")\n        return device\n\n    print("CUDA not available, using CPU.")\n    return torch.device("cpu")\n\n\nclass NNLM(nn.Module):\n    def __init__(self, V, m=100, h=100, n=4, direct=False):\n        super().__init__()\n        self.V = V\n        self.m = m\n        self.h = h\n        self.n = n\n        self.direct = direct\n        self.context_dim = (n - 1) * m\n\n        self.C = nn.Embedding(V, m)\n        self.H = nn.Linear(self.context_dim, h, bias=False)\n        self.d = nn.Parameter(torch.zeros(h))\n        self.U = nn.Linear(h, V, bias=False)\n        self.W = nn.Linear(self.context_dim, V, bias=False) if direct else None\n        self.b = nn.Parameter(torch.zeros(V))\n\n        self.reset_parameters()\n\n    def reset_parameters(self):\n        nn.init.uniform_(self.C.weight, -0.1, 0.1)\n        nn.init.xavier_uniform_(self.H.weight)\n        nn.init.xavier_uniform_(self.U.weight)\n        if self.W is not None:\n            nn.init.xavier_uniform_(self.W.weight)\n        nn.init.zeros_(self.d)\n        nn.init.zeros_(self.b)\n\n    def forward(self, context):\n        x = self.C(context).reshape(context.size(0), -1)\n        hidden_pre = self.H(x) + self.d\n        hidden = torch.tanh(hidden_pre)\n        hidden_term = self.U(hidden)\n        direct_term = self.W(x) if self.W is not None else 0.0\n        return self.b + direct_term + hidden_term\n\n    def embeddings(self):\n        return self.C.weight.detach().cpu().numpy().astype(np.float32).copy()\n\n\ndef batches(X, Y, batch_size, device, shuffle=True):\n    size = len(Y)\n    if size == 0:\n        return\n    order = np.random.permutation(size) if shuffle else np.arange(size)\n    for start in range(0, size, batch_size):\n        indices = order[start : start + batch_size]\n        xb = torch.from_numpy(X[indices]).long().to(device, non_blocking=True)\n        yb = torch.from_numpy(Y[indices]).long().to(device, non_blocking=True)\n        yield xb, yb\n\n\n@torch.no_grad()\ndef mean_nll(model, X, Y, device, batch_size=2048):\n    model.eval()\n    total_nll = 0.0\n    total_items = 0\n    for xb, yb in batches(X, Y, batch_size=batch_size, device=device, shuffle=False):\n        logits = model(xb)\n        loss_sum = F.cross_entropy(logits, yb, reduction="sum")\n        total_nll += float(loss_sum.detach().cpu())\n        total_items += int(yb.size(0))\n    if total_items == 0:\n        return float("inf")\n    return total_nll / total_items\n\n\n@torch.no_grad()\ndef perplexity(model, X, Y, device, batch_size=2048):\n    return float(np.exp(min(mean_nll(model, X, Y, device=device, batch_size=batch_size), 50.0)))\n\n\ndef optimizer_to_device(optimizer, device):\n    for state in optimizer.state.values():\n        for key, value in list(state.items()):\n            if torch.is_tensor(value):\n                state[key] = value.to(device)\n\n\ndef save_model_checkpoint(lang, model, config, stoi, itos):\n    files = q5_files(lang)\n    cpu_state = {name: tensor.detach().cpu() for name, tensor in model.state_dict().items()}\n    torch.save(\n        {\n            "state_dict": cpu_state,\n            "config": {\n                "V": len(itos),\n                "m": config["m"],\n                "h": config["h"],\n                "n": config["n"],\n                "direct": config["direct"],\n            },\n            "stoi": stoi,\n            "itos": itos,\n        },\n        files["model"],\n    )\n\n\ndef save_resume_checkpoint(\n    lang,\n    epoch,\n    model,\n    optimizer,\n    history,\n    best_state,\n    best_valid_nll,\n    stale_epochs,\n    config,\n    itos,\n):\n    files = q5_files(lang)\n    optimizer_state = copy.deepcopy(optimizer.state_dict())\n    for state in optimizer_state.get("state", {}).values():\n        for key, value in list(state.items()):\n            if torch.is_tensor(value):\n                state[key] = value.detach().cpu()\n\n    torch.save(\n        {\n            "epoch": int(epoch),\n            "model_state_dict": {name: tensor.detach().cpu() for name, tensor in model.state_dict().items()},\n            "optimizer_state_dict": optimizer_state,\n            "history": history,\n            "best_state_dict": {name: tensor.detach().cpu() for name, tensor in best_state.items()},\n            "best_valid_nll": float(best_valid_nll),\n            "stale_epochs": int(stale_epochs),\n            "config": {\n                "n": config["n"],\n                "m": config["m"],\n                "h": config["h"],\n                "direct": config["direct"],\n                "epochs": config["epochs"],\n                "batch_size": config["batch_size"],\n                "lr": config["lr"],\n                "weight_decay": config["weight_decay"],\n                "patience": config["patience"],\n                "target_vocab_size": config["target_vocab_size"],\n            },\n            "itos": itos,\n        },\n        files["checkpoint"],\n    )\n    print(f"[checkpoint] saved -> {files[\'checkpoint\'].name} (epoch {epoch})")\n\n\ndef load_resume_checkpoint(lang, model, optimizer, device, itos, config):\n    files = q5_files(lang)\n    if not files["checkpoint"].exists():\n        return 0, None\n\n    ckpt = torch.load(files["checkpoint"], map_location="cpu")\n    if ckpt.get("itos") != itos:\n        raise ValueError(f"[checkpoint] vocabulary mismatch for {lang}.")\n\n    saved_cfg = ckpt.get("config", {})\n    for key in ("n", "m", "h", "direct", "target_vocab_size"):\n        if saved_cfg.get(key) != config.get(key):\n            raise ValueError(f"[checkpoint] config mismatch for {lang} on \'{key}\'.")\n\n    model.load_state_dict(ckpt["model_state_dict"])\n    optimizer.load_state_dict(ckpt["optimizer_state_dict"])\n    optimizer_to_device(optimizer, device)\n    return int(ckpt.get("epoch", 0)), ckpt\n\n\ndef train_model(model, resources, device, config, lang):\n    decay_params = [model.C.weight, model.H.weight, model.U.weight]\n    if model.W is not None:\n        decay_params.append(model.W.weight)\n\n    optimizer = optim.Adam(\n        [\n            {"params": decay_params, "weight_decay": config["weight_decay"]},\n            {"params": [model.d, model.b], "weight_decay": 0.0},\n        ],\n        lr=config["lr"],\n    )\n\n    history = {\n        "train_loss": [],\n        "train_ppl_token": [],\n        "valid_loss": [],\n        "valid_ppl_token": [],\n        "epoch_seconds": [],\n        "elapsed_seconds": 0.0,\n        "best_epoch": 0,\n        "best_valid_nll": float("inf"),\n        "converged": False,\n        "stopped_epoch": 0,\n    }\n\n    start_epoch = 0\n    stale_epochs = 0\n    best_state = copy.deepcopy(model.state_dict())\n\n    start_epoch, resume_state = load_resume_checkpoint(\n        lang=lang,\n        model=model,\n        optimizer=optimizer,\n        device=device,\n        itos=resources["itos"],\n        config=config,\n    )\n    if resume_state is not None:\n        history = resume_state["history"]\n        history.setdefault("epoch_seconds", [])\n        history.setdefault("elapsed_seconds", 0.0)\n        history["best_valid_nll"] = resume_state.get("best_valid_nll", history["best_valid_nll"])\n        stale_epochs = int(resume_state.get("stale_epochs", 0))\n        best_state = resume_state["best_state_dict"]\n        print(\n            f"[checkpoint] resumed {lang} from epoch {start_epoch} "\n            f"(best valid nll {history[\'best_valid_nll\']:.4f}, "\n            f"elapsed {format_duration(history[\'elapsed_seconds\'])})"\n        )\n\n    total_batches = max(1, math.ceil(len(resources["Ytr"]) / max(config["batch_size"], 1)))\n    print(\n        f"Training setup | epochs={config[\'epochs\']} | batch_size={config[\'batch_size\']} | "\n        f"train_targets={len(resources[\'Ytr\']):,} | valid_targets={len(resources[\'Yva\']):,} | "\n        f"batches/epoch={total_batches}",\n        flush=True,\n    )\n\n    for epoch in range(start_epoch + 1, config["epochs"] + 1):\n        epoch_started = time.time()\n        progress_step = max(1, total_batches // 20)\n        total_nll = 0.0\n        total_items = 0\n\n        model.train()\n        for batch_idx, (xb, yb) in enumerate(\n            batches(resources["Xtr"], resources["Ytr"], batch_size=config["batch_size"], device=device, shuffle=True),\n            start=1,\n        ):\n            optimizer.zero_grad(set_to_none=True)\n            logits = model(xb)\n            loss = F.cross_entropy(logits, yb, reduction="mean")\n            loss.backward()\n            optimizer.step()\n\n            batch_items = int(yb.size(0))\n            total_items += batch_items\n            total_nll += float(loss.detach().cpu()) * batch_items\n\n            if batch_idx == 1 or batch_idx % progress_step == 0 or batch_idx == total_batches:\n                batch_elapsed = time.time() - epoch_started\n                epoch_eta = (total_batches - batch_idx) * (batch_elapsed / max(batch_idx, 1))\n                print(\n                    f"[progress] epoch {epoch:02d}/{config[\'epochs\']} | "\n                    f"batch {batch_idx:04d}/{total_batches} ({100.0 * batch_idx / total_batches:5.1f}%) | "\n                    f"epoch elapsed {format_duration(batch_elapsed)} | epoch ETA {format_duration(epoch_eta)}",\n                    flush=True,\n                )\n\n        train_loss = total_nll / max(total_items, 1)\n        valid_loss = mean_nll(\n            model,\n            resources["Xva"],\n            resources["Yva"],\n            device=device,\n            batch_size=config["batch_size"],\n        )\n        train_ppl_token = float(np.exp(min(train_loss, 50.0)))\n        valid_ppl_token = float(np.exp(min(valid_loss, 50.0)))\n\n        epoch_seconds = time.time() - epoch_started\n        history["train_loss"].append(train_loss)\n        history["train_ppl_token"].append(train_ppl_token)\n        history["valid_loss"].append(valid_loss)\n        history["valid_ppl_token"].append(valid_ppl_token)\n        history["epoch_seconds"].append(epoch_seconds)\n        history["elapsed_seconds"] += epoch_seconds\n\n        if valid_loss + 1e-8 < history["best_valid_nll"]:\n            history["best_valid_nll"] = valid_loss\n            history["best_epoch"] = epoch\n            best_state = copy.deepcopy(model.state_dict())\n            stale_epochs = 0\n        else:\n            stale_epochs += 1\n\n        remaining_epochs = max(config["epochs"] - epoch, 0)\n        avg_epoch = history["elapsed_seconds"] / max(len(history["epoch_seconds"]), 1)\n        print(\n            f"Epoch {epoch:02d} | train loss={train_loss:.4f} | train ppl(token)={train_ppl_token:.2f} | "\n            f"valid loss={valid_loss:.4f} | valid ppl(token)={valid_ppl_token:.2f} | "\n            f"best epoch={history[\'best_epoch\'] or epoch} | elapsed={format_duration(history[\'elapsed_seconds\'])} | "\n            f"remaining~={format_duration(remaining_epochs * avg_epoch)}",\n            flush=True,\n        )\n\n        if epoch % CHECKPOINT_EVERY == 0 or epoch == config["epochs"]:\n            save_resume_checkpoint(\n                lang=lang,\n                epoch=epoch,\n                model=model,\n                optimizer=optimizer,\n                history=history,\n                best_state=best_state,\n                best_valid_nll=history["best_valid_nll"],\n                stale_epochs=stale_epochs,\n                config=config,\n                itos=resources["itos"],\n            )\n\n        if stale_epochs >= config["patience"]:\n            history["converged"] = True\n            history["stopped_epoch"] = epoch\n            break\n\n    if not history["stopped_epoch"]:\n        history["stopped_epoch"] = len(history["train_loss"])\n\n    model.load_state_dict(best_state)\n    return history\n\n\ndef count_parameters(V, m, h, n, direct=False):\n    context_dim = (n - 1) * m\n    counts = {\n        "C": V * m,\n        "H": h * context_dim,\n        "d": h,\n        "U": V * h,\n        "W": V * context_dim if direct else 0,\n        "b": V,\n    }\n    counts["TOTAL"] = sum(counts.values())\n    counts["QUESTION_FORMULA_IF_DIRECT"] = V * (1 + n * m + h) + h * (1 + context_dim)\n    counts["FORMULA_WITH_CURRENT_SETTING"] = (\n        V * (1 + m + h + (context_dim if direct else 0)) + h * (1 + context_dim)\n    )\n    return counts\n\n\ndef normalized_ppl_from_mean_nll(mean_loss, target_count, word_units):\n    if not math.isfinite(mean_loss) or target_count <= 0 or word_units <= 0:\n        return float("inf")\n    total_nll = float(mean_loss) * float(target_count)\n    return float(np.exp(min(total_nll / float(word_units), 50.0)))\n\n\ndef save_figure(fig, path):\n    path = Path(path)\n    fig.tight_layout()\n    fig.savefig(path, dpi=180, bbox_inches="tight")\n    plt.close(fig)\n    return str(path)\n\n\ndef plot_convergence(history, resources, lang, save_path):\n    train_stats = resources["train_stats"]\n    valid_stats = resources["valid_stats"]\n    train_word_ppl = [\n        normalized_ppl_from_mean_nll(loss, train_stats["subword_targets"], train_stats["surface_word_units"])\n        for loss in history["train_loss"]\n    ]\n    valid_word_ppl = [\n        normalized_ppl_from_mean_nll(loss, valid_stats["subword_targets"], valid_stats["surface_word_units"])\n        for loss in history["valid_loss"]\n    ]\n    epochs = np.arange(1, len(history["train_loss"]) + 1)\n\n    fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))\n    axes[0].plot(epochs, history["train_loss"], marker="o", label="train loss")\n    axes[0].plot(epochs, history["valid_loss"], marker="o", label="valid loss")\n    axes[0].axvline(history["best_epoch"], color="tab:red", linestyle="--", alpha=0.7)\n    axes[0].set_title(f"{language_label(lang)} | Cross-Entropy")\n    axes[0].set_xlabel("Epoch")\n    axes[0].set_ylabel("Mean NLL per subword target")\n    axes[0].legend()\n\n    axes[1].plot(epochs, train_word_ppl, marker="o", label="train ppl / word-unit")\n    axes[1].plot(epochs, valid_word_ppl, marker="o", label="valid ppl / word-unit")\n    axes[1].axvline(history["best_epoch"], color="tab:red", linestyle="--", alpha=0.7)\n    axes[1].set_title(f"{language_label(lang)} | Per-Word Normalized Perplexity")\n    axes[1].set_xlabel("Epoch")\n    axes[1].set_ylabel("Perplexity")\n    axes[1].legend()\n    return save_figure(fig, save_path)\n\n\ndef json_ready(value):\n    if isinstance(value, (np.integer, np.floating)):\n        return value.item()\n    if isinstance(value, np.ndarray):\n        return value.tolist()\n    if isinstance(value, pd.DataFrame):\n        return value.to_dict(orient="records")\n    if isinstance(value, Path):\n        return str(value)\n    if isinstance(value, dict):\n        return {str(key): json_ready(val) for key, val in value.items()}\n    if isinstance(value, list):\n        return [json_ready(val) for val in value]\n    return value\n\n\ndef save_json(path, data):\n    Path(path).write_text(\n        json.dumps(json_ready(data), indent=2, ensure_ascii=False),\n        encoding="utf-8",\n    )\n\n\ndef public_config(config: dict):\n    return {key: value for key, value in config.items() if key != "hf_token"}\n\n\ndef find_local_variant(*names):\n    for name in names:\n        if not name:\n            continue\n        path = Path(name)\n        if path.exists():\n            return path\n    return None\n\n\ndef load_q4_metric_summary(lang):\n    json_path = find_local_variant(\n        f"q4_metrics_summary_{lang}.json",\n        f"q4_metrics_summary_{lang}",\n    )\n    if json_path is not None:\n        data = json.loads(json_path.read_text(encoding="utf-8"))\n        best_ngram = min(\n            float(data["perplexities"]["laplace_ngram"]),\n            float(data["perplexities"]["interp_ngram"]),\n        )\n        return {\n            "source": json_path.name,\n            "word_nnlm_ppl": float(data["perplexities"]["nnlm"]),\n            "best_ngram_ppl": best_ngram,\n            "laplace_ppl": float(data["perplexities"]["laplace_ngram"]),\n            "interp_ppl": float(data["perplexities"]["interp_ngram"]),\n        }\n\n    csv_path = find_local_variant(\n        f"q4_perplexity_table_{lang}.csv",\n        f"q4_perplexity_table_{lang}",\n    )\n    if csv_path is not None:\n        df = pd.read_csv(csv_path)\n        row_map = {str(row["model"]).strip(): float(row["perplexity"]) for _, row in df.iterrows()}\n        nnlm = row_map.get("NNLM")\n        laplace = row_map.get("Laplace n-gram")\n        interp = row_map.get("Interpolated n-gram")\n        if nnlm is not None and laplace is not None and interp is not None:\n            return {\n                "source": csv_path.name,\n                "word_nnlm_ppl": nnlm,\n                "best_ngram_ppl": min(laplace, interp),\n                "laplace_ppl": laplace,\n                "interp_ppl": interp,\n            }\n    return None\n\n\ndef load_q4_model_artifact(lang):\n    path = find_local_variant(f"nnlm_{lang}.pt", f"nnlm_{lang}")\n    if path is None:\n        return None\n\n    ckpt = torch.load(path, map_location="cpu")\n    embeddings = np.asarray(ckpt["state_dict"]["C.weight"], dtype=np.float32)\n    itos = list(ckpt["itos"])\n    return {\n        "source": path.name,\n        "embeddings": embeddings,\n        "stoi": ckpt["stoi"],\n        "itos": itos,\n        "config": ckpt["config"],\n    }\n\n\ndef cosine(u, v):\n    u = np.asarray(u, dtype=np.float32)\n    v = np.asarray(v, dtype=np.float32)\n    denom = np.linalg.norm(u) * np.linalg.norm(v)\n    if denom <= 1e-12:\n        return 0.0\n    return float(np.dot(u, v) / denom)\n\n\ndef is_content_word(token):\n    if token in SPECIAL_SET:\n        return False\n    return re.search(r"\\w", str(token), flags=re.UNICODE) is not None\n\n\ndef build_embedding_comparison(lang, resources, model_embeddings, q4_model):\n    files = q5_files(lang)\n    if q4_model is None:\n        empty = pd.DataFrame(\n            [\n                {\n                    "status": "missing_q4_model",\n                    "note": "Upload nnlm_<lang>.pt from Question 4 to enable embedding comparisons.",\n                }\n            ]\n        )\n        empty.to_csv(files["embedding_compare"], index=False)\n        return {\n            "status": "missing_q4_model",\n            "rows": 0,\n        }\n\n    lower_counts = resources["surface_stats"]["lower_counts"]\n    representatives = resources["surface_stats"]["representatives"]\n    tokenizer = resources["tokenizer"]\n    q4_stoi = q4_model["stoi"]\n    q4_embeddings = q4_model["embeddings"]\n\n    candidates = [\n        word\n        for word, _ in sorted(lower_counts.items(), key=lambda item: (-item[1], item[0]))\n        if word in q4_stoi and is_content_word(word)\n    ]\n\n    rows = []\n    for word in candidates[:300]:\n        surface_form = representatives.get(word, word)\n        pieces = tokenizer.encode_word(surface_form)\n        if not pieces:\n            continue\n\n        piece_ids = [resources["stoi"][piece] for piece in pieces]\n        whole_word = len(pieces) == 1 and token_to_surface_word(pieces[0]) == surface_form\n        direct_cosine = None\n        if whole_word:\n            direct_cosine = cosine(q4_embeddings[q4_stoi[word]], model_embeddings[piece_ids[0]])\n\n        proxy_vector = model_embeddings[piece_ids].mean(axis=0)\n        proxy_cosine = cosine(q4_embeddings[q4_stoi[word]], proxy_vector)\n        rows.append(\n            {\n                "word_q4": word,\n                "surface_form_q5": surface_form,\n                "frequency": int(lower_counts[word]),\n                "subword_count": len(pieces),\n                "subword_tokens": " | ".join(pieces),\n                "is_whole_token": bool(whole_word),\n                "direct_cosine_if_whole_token": direct_cosine,\n                "mean_subword_cosine_proxy": proxy_cosine,\n            }\n        )\n\n    df = pd.DataFrame(rows)\n    if df.empty:\n        pd.DataFrame(\n            [\n                {\n                    "status": "no_overlap",\n                    "note": "No overlapping content words were found between Q4 and the subword model.",\n                }\n            ]\n        ).to_csv(files["embedding_compare"], index=False)\n        return {\n            "status": "no_overlap",\n            "rows": 0,\n        }\n\n    df.to_csv(files["embedding_compare"], index=False)\n    whole = df[df["is_whole_token"] & df["direct_cosine_if_whole_token"].notna()].copy()\n    split = df[df["subword_count"] > 1].copy()\n\n    def mean_or_none(series):\n        series = pd.to_numeric(series, errors="coerce").dropna()\n        return float(series.mean()) if not series.empty else None\n\n    whole_frequent = whole.sort_values(["frequency", "word_q4"], ascending=[False, True]).head(50)\n    split_frequent = split.sort_values(["frequency", "word_q4"], ascending=[False, True]).head(50)\n\n    return {\n        "status": "ok",\n        "rows": int(len(df)),\n        "whole_token_rows": int(len(whole)),\n        "split_rows": int(len(split)),\n        "mean_whole_token_cosine": mean_or_none(whole["direct_cosine_if_whole_token"]),\n        "mean_split_proxy_cosine": mean_or_none(split["mean_subword_cosine_proxy"]),\n        "mean_top50_whole_token_cosine": mean_or_none(whole_frequent["direct_cosine_if_whole_token"]),\n        "mean_top50_split_proxy_cosine": mean_or_none(split_frequent["mean_subword_cosine_proxy"]),\n        "csv": str(files["embedding_compare"]),\n    }\n\n\ndef build_written_answers(lang, analysis_summary):\n    files = q5_files(lang)\n    q4_metrics = analysis_summary.get("q4_metrics")\n    q4_model = analysis_summary.get("q4_model")\n    embed = analysis_summary.get("embedding_comparison", {})\n    ppl = analysis_summary["perplexities"]\n    params = analysis_summary["parameter_comparison"]\n    split_stats = analysis_summary["split_stats"]\n\n    lines = [\n        f"# Question 5 draft for {language_label(lang)}",\n        "",\n        "## 1. Perplexity",\n        (\n            f"- Subword NNLM test perplexity (per-word normalized) = "\n            f"{ppl[\'subword_per_word_normalized\']:.4f}."\n        ),\n        (\n            f"- Raw subword-token perplexity = {ppl[\'subword_token_level\']:.4f}; "\n            f"average subwords per surface token on the test split = "\n            f"{split_stats[\'test\'][\'avg_subwords_per_surface_token\']:.4f}."\n        ),\n    ]\n\n    if q4_metrics is not None:\n        lines.append(\n            f"- Word-level Q4 NNLM perplexity = {q4_metrics[\'word_nnlm_ppl\']:.4f}; "\n            f"best Q4 n-gram perplexity = {q4_metrics[\'best_ngram_ppl\']:.4f}."\n        )\n        lines.append(\n            "- The subword model should usually trade a smaller vocabulary and near-zero OOV risk "\n            "for longer prediction sequences."\n        )\n    else:\n        lines.append(\n            "- Upload `q4_metrics_summary_<lang>.json` or `q4_perplexity_table_<lang>.csv` if you want "\n            "the notebook to fill in the exact comparison against your earlier word-level results."\n        )\n\n    lines.extend(\n        [\n            "",\n            "## 2. Embedding Cosine Similarity",\n        ]\n    )\n    if embed.get("status") == "ok":\n        lines.append(\n            f"- Whole-token overlap rows = {embed[\'whole_token_rows\']}; "\n            f"mean cosine on those rows = {fmt_optional(embed[\'mean_whole_token_cosine\'])}."\n        )\n        lines.append(\n            f"- Split-word rows = {embed[\'split_rows\']}; "\n            f"proxy cosine from the mean of subword pieces = {fmt_optional(embed[\'mean_split_proxy_cosine\'])}."\n        )\n        lines.append(\n            "- Frequent whole-word tokens usually stay closer across the two spaces because the "\n            "subword model can dedicate one vector to the full form, while split words distribute "\n            "their signal across multiple pieces."\n        )\n    else:\n        lines.append(\n            "- Upload `nnlm_<lang>.pt` from Question 4 to enable the embedding-space comparison."\n        )\n\n    lines.extend(\n        [\n            "",\n            "## 3. Sparsity and Parameter Count",\n            (\n                f"- Subword vocabulary size = {params[\'subword_vocab_size\']:,}; "\n                f"subword NNLM total parameters = {params[\'subword_total_params\']:,}."\n            ),\n        ]\n    )\n    if q4_model is not None:\n        lines.append(\n            f"- Word-level vocabulary size = {params[\'word_vocab_size\']:,}; "\n            f"word-level NNLM total parameters = {params[\'word_total_params\']:,}."\n        )\n        lines.append(\n            "- The subword model reduces vocabulary-driven parameter growth, but it usually needs "\n            "more prediction steps per sentence because many words split into multiple pieces."\n        )\n    else:\n        lines.append(\n            "- Upload `nnlm_<lang>.pt` from Question 4 if you want the exact parameter comparison filled in automatically."\n        )\n\n    files["written_answers"].write_text("\\n".join(lines) + "\\n", encoding="utf-8")\n    return files["written_answers"]\n\n\ndef fmt_optional(value):\n    return "nan" if value is None else f"{value:.4f}"\n\n\ndef build_analysis_outputs(lang, resources, model, history, config):\n    files = q5_files(lang)\n    device = next(model.parameters()).device\n\n    test_nll = mean_nll(model, resources["Xte"], resources["Yte"], device=device, batch_size=config["batch_size"])\n    test_ppl_token = float(np.exp(min(test_nll, 50.0)))\n    test_ppl_word = normalized_ppl_from_mean_nll(\n        test_nll,\n        resources["test_stats"]["subword_targets"],\n        resources["test_stats"]["surface_word_units"],\n    )\n\n    train_log = {\n        "test_ppl": test_ppl_word,\n        "test_ppl_token_level": test_ppl_token,\n        "train_ppl_history": [\n            normalized_ppl_from_mean_nll(\n                loss,\n                resources["train_stats"]["subword_targets"],\n                resources["train_stats"]["surface_word_units"],\n            )\n            for loss in history["train_loss"]\n        ],\n        "valid_ppl_history": [\n            normalized_ppl_from_mean_nll(\n                loss,\n                resources["valid_stats"]["subword_targets"],\n                resources["valid_stats"]["surface_word_units"],\n            )\n            for loss in history["valid_loss"]\n        ],\n        "raw_train_token_ppl_history": history["train_ppl_token"],\n        "raw_valid_token_ppl_history": history["valid_ppl_token"],\n        "history": history,\n        "split_stats": {\n            "train": resources["train_stats"],\n            "valid": resources["valid_stats"],\n            "test": resources["test_stats"],\n        },\n    }\n    save_json(files["train_log"], train_log)\n\n    q4_metrics = load_q4_metric_summary(lang)\n    q4_model = load_q4_model_artifact(lang)\n\n    perplexity_rows = [\n        {\n            "model": "Subword NNLM",\n            "perplexity": test_ppl_word,\n            "normalization": "per surface token + sentence end",\n            "source": files["train_log"].name,\n        },\n        {\n            "model": "Subword NNLM (token-level)",\n            "perplexity": test_ppl_token,\n            "normalization": "per subword target",\n            "source": files["train_log"].name,\n        },\n    ]\n    if q4_metrics is not None:\n        perplexity_rows.append(\n            {\n                "model": "Word-level NNLM (Q4)",\n                "perplexity": q4_metrics["word_nnlm_ppl"],\n                "normalization": "word-level",\n                "source": q4_metrics["source"],\n            }\n        )\n        perplexity_rows.append(\n            {\n                "model": "Best n-gram baseline (Q4)",\n                "perplexity": q4_metrics["best_ngram_ppl"],\n                "normalization": "word-level",\n                "source": q4_metrics["source"],\n            }\n        )\n    pd.DataFrame(perplexity_rows).to_csv(files["perplexity_compare"], index=False)\n\n    subword_counts = count_parameters(\n        V=len(resources["itos"]),\n        m=config["m"],\n        h=config["h"],\n        n=config["n"],\n        direct=config["direct"],\n    )\n    parameter_rows = [\n        {\n            "model": "Subword NNLM",\n            "lang": lang,\n            "language": language_label(lang),\n            "vocab_size": len(resources["itos"]),\n            "embedding_params": subword_counts["C"],\n            "total_params": subword_counts["TOTAL"],\n            "n": config["n"],\n            "m": config["m"],\n            "h": config["h"],\n            "direct": config["direct"],\n            "source": files["model"].name,\n        }\n    ]\n    parameter_summary = {\n        "subword_vocab_size": int(len(resources["itos"])),\n        "subword_total_params": int(subword_counts["TOTAL"]),\n        "subword_embedding_params": int(subword_counts["C"]),\n        "word_vocab_size": None,\n        "word_total_params": None,\n        "word_embedding_params": None,\n    }\n\n    if q4_model is not None:\n        word_cfg = q4_model["config"]\n        word_counts = count_parameters(\n            V=int(word_cfg["V"]),\n            m=int(word_cfg["m"]),\n            h=int(word_cfg["h"]),\n            n=int(word_cfg["n"]),\n            direct=bool(word_cfg["direct"]),\n        )\n        parameter_rows.append(\n            {\n                "model": "Word-level NNLM (Q4)",\n                "lang": lang,\n                "language": language_label(lang),\n                "vocab_size": int(word_cfg["V"]),\n                "embedding_params": word_counts["C"],\n                "total_params": word_counts["TOTAL"],\n                "n": int(word_cfg["n"]),\n                "m": int(word_cfg["m"]),\n                "h": int(word_cfg["h"]),\n                "direct": bool(word_cfg["direct"]),\n                "source": q4_model["source"],\n            }\n        )\n        parameter_summary["word_vocab_size"] = int(word_cfg["V"])\n        parameter_summary["word_total_params"] = int(word_counts["TOTAL"])\n        parameter_summary["word_embedding_params"] = int(word_counts["C"])\n\n    pd.DataFrame(parameter_rows).to_csv(files["parameter_compare"], index=False)\n\n    embeddings = model.embeddings()\n    embedding_summary = build_embedding_comparison(\n        lang=lang,\n        resources=resources,\n        model_embeddings=embeddings,\n        q4_model=q4_model,\n    )\n\n    analysis_summary = {\n        "lang": lang,\n        "language": language_label(lang),\n        "config": {\n            "n": config["n"],\n            "m": config["m"],\n            "h": config["h"],\n            "epochs": config["epochs"],\n            "batch_size": config["batch_size"],\n            "lr": config["lr"],\n            "weight_decay": config["weight_decay"],\n            "patience": config["patience"],\n            "direct": config["direct"],\n            "target_vocab_size": config["target_vocab_size"],\n        },\n        "tokenizer_summary": resources["tokenizer"].summary,\n        "split_stats": {\n            "train": resources["train_stats"],\n            "valid": resources["valid_stats"],\n            "test": resources["test_stats"],\n        },\n        "perplexities": {\n            "subword_token_level": test_ppl_token,\n            "subword_per_word_normalized": test_ppl_word,\n        },\n        "q4_metrics": q4_metrics,\n        "q4_model": {\n            "source": q4_model["source"],\n            "config": q4_model["config"],\n        }\n        if q4_model is not None\n        else None,\n        "parameter_comparison": parameter_summary,\n        "embedding_comparison": embedding_summary,\n        "files": {\n            "train_log": str(files["train_log"]),\n            "perplexity_compare": str(files["perplexity_compare"]),\n            "parameter_compare": str(files["parameter_compare"]),\n            "embedding_compare": str(files["embedding_compare"]),\n        },\n    }\n    save_json(files["analysis_summary"], analysis_summary)\n    build_written_answers(lang, analysis_summary)\n    return train_log, analysis_summary\n\n\ndef bundle_files(bundle_name: str | Path, members):\n    bundle_path = Path(bundle_name)\n    with zipfile.ZipFile(bundle_path, "w", zipfile.ZIP_DEFLATED) as zf:\n        for member in members:\n            path = Path(member)\n            if path.exists():\n                zf.write(path, arcname=path.name)\n    print(f"Created {bundle_path.name}")\n    return bundle_path\n\n\ndef build_language_bundle(lang):\n    files = q5_files(lang)\n    members = [\n        files["tokenizer"],\n        files["model"],\n        files["checkpoint"],\n        files["train_log"],\n        files["train_state"],\n        files["train_summary"],\n        files["bpe_summary"],\n        files["plot"],\n        files["perplexity_compare"],\n        files["parameter_compare"],\n        files["embedding_compare"],\n        files["analysis_summary"],\n        files["written_answers"],\n        cache_path_for_lang(lang),\n    ]\n    return bundle_files(files["bundle"], members)\n\n\ndef build_problem_5_bundle_if_ready():\n    required = [\n        Path("bpe_en.json"),\n        Path("bpe_hi.json"),\n        Path("bpe_nnlm_en.pt"),\n        Path("bpe_nnlm_hi.pt"),\n        Path("bpe_train_log_en.json"),\n        Path("bpe_train_log_hi.json"),\n    ]\n    missing = [path.name for path in required if not path.exists()]\n    if missing:\n        print("Combined Q5 zip not created yet. Missing:", ", ".join(missing))\n        return None\n    bundle_files("problem_5.zip", required)\n    bundle_files("question_5.zip", required)\n    return Path("problem_5.zip")\n\n\ndef maybe_build_combined_written_answers():\n    needed = [q5_files(lang)["analysis_summary"] for lang in LANGS]\n    if not all(path.exists() for path in needed):\n        return None\n\n    summaries = {\n        lang: json.loads(q5_files(lang)["analysis_summary"].read_text(encoding="utf-8"))\n        for lang in LANGS\n    }\n    lines = ["# Question 5 written-answer draft", ""]\n    for lang in LANGS:\n        summary = summaries[lang]\n        ppl = summary["perplexities"]\n        params = summary["parameter_comparison"]\n        embed = summary["embedding_comparison"]\n        lines.extend(\n            [\n                f"## {language_label(lang)}",\n                (\n                    f"- Subword NNLM test perplexity (per-word normalized): "\n                    f"{ppl[\'subword_per_word_normalized\']:.4f}"\n                ),\n                (\n                    f"- Subword NNLM token-level perplexity: "\n                    f"{ppl[\'subword_token_level\']:.4f}"\n                ),\n            ]\n        )\n        if summary["q4_metrics"] is not None:\n            lines.append(\n                f"- Word-level Q4 NNLM: {summary[\'q4_metrics\'][\'word_nnlm_ppl\']:.4f}; "\n                f"best Q4 n-gram: {summary[\'q4_metrics\'][\'best_ngram_ppl\']:.4f}"\n            )\n        if params["word_total_params"] is not None:\n            lines.append(\n                f"- Subword params: {params[\'subword_total_params\']:,}; "\n                f"word-level params: {params[\'word_total_params\']:,}"\n            )\n        if embed.get("status") == "ok":\n            lines.append(\n                f"- Whole-token cosine mean: {fmt_optional(embed[\'mean_whole_token_cosine\'])}; "\n                f"split-word proxy cosine mean: {fmt_optional(embed[\'mean_split_proxy_cosine\'])}"\n            )\n        lines.append("")\n\n    output = Path("Q5_WRITTEN_ANSWERS.md")\n    output.write_text("\\n".join(lines).strip() + "\\n", encoding="utf-8")\n    return output\n\n\ndef extract_bundle_if_present(bundle_name):\n    path = Path(bundle_name)\n    if path.exists():\n        with zipfile.ZipFile(path, "r") as zf:\n            zf.extractall(".")\n        print(f"Extracted {path.name}")\n\n\ndef stage_kaggle_input_files(required_names):\n    input_root = Path("/kaggle/input")\n    if not input_root.exists():\n        return\n\n    for name in required_names:\n        local_path = Path(name)\n        if local_path.exists():\n            continue\n        matches = list(input_root.rglob(name))\n        if not matches:\n            continue\n        shutil.copy2(matches[0], local_path)\n        print(f"Copied input file {matches[0]} -> {local_path}")\n\n\ndef run_full_language_pipeline(lang: str, config: dict, device_id: int | None = None):\n    set_shared_seed()\n    files = q5_files(lang)\n    expected_inputs = [\n        EN_CACHE,\n        HI_CACHE,\n        files["tokenizer"].name,\n        files["checkpoint"].name,\n        f"q4_{lang}_artifacts.zip",\n        "question_4_bundle.zip",\n        f"nnlm_{lang}.pt",\n        f"q4_metrics_summary_{lang}.json",\n        f"q4_metrics_summary_{lang}",\n        f"q4_perplexity_table_{lang}.csv",\n        f"q4_perplexity_table_{lang}",\n    ]\n    stage_kaggle_input_files(expected_inputs)\n    extract_bundle_if_present(f"q4_{lang}_artifacts.zip")\n    extract_bundle_if_present("question_4_bundle.zip")\n\n    if device_id is None:\n        device_id = 0 if torch.cuda.is_available() else -1\n    device = resolve_device(device_id)\n\n    print(f"Running full Q5 pipeline for {language_label(lang)} on device_id={device_id}")\n    resources = prepare_language_resources(lang, config)\n\n    model = NNLM(\n        V=len(resources["itos"]),\n        m=config["m"],\n        h=config["h"],\n        n=config["n"],\n        direct=config["direct"],\n    ).to(device)\n\n    history = train_model(model, resources=resources, device=device, config=config, lang=lang)\n    save_model_checkpoint(lang, model, config, resources["stoi"], resources["itos"])\n    plot_path = plot_convergence(history, resources, lang=lang, save_path=files["plot"])\n\n    train_summary = {\n        "lang": lang,\n        "language": language_label(lang),\n        "config": public_config(config),\n        "tokenizer_summary": resources["tokenizer"].summary,\n        "split_stats": {\n            "train": resources["train_stats"],\n            "valid": resources["valid_stats"],\n            "test": resources["test_stats"],\n        },\n        "history": history,\n        "plot_path": plot_path,\n        "vocab_size": len(resources["itos"]),\n    }\n    with files["train_state"].open("wb") as handle:\n        pickle.dump(train_summary, handle)\n    save_json(files["train_summary"], train_summary)\n\n    build_analysis_outputs(lang, resources, model, history, config)\n    bundle = build_language_bundle(lang)\n    combined = build_problem_5_bundle_if_ready()\n    written = maybe_build_combined_written_answers()\n\n    print(f"Completed Q5 pipeline for {language_label(lang)}")\n    print("Artifacts:")\n    for path in [\n        files["tokenizer"],\n        files["model"],\n        files["checkpoint"],\n        files["train_log"],\n        files["plot"],\n        files["perplexity_compare"],\n        files["parameter_compare"],\n        files["embedding_compare"],\n        files["analysis_summary"],\n        files["written_answers"],\n        bundle,\n        combined,\n        written,\n    ]:\n        if path is not None and Path(path).exists():\n            print(" -", Path(path).name)\n', encoding="utf-8")
print("Prepared q5_parallel_runtime.py in the current session.")


Prepared q5_parallel_runtime.py in the current session.


## Runtime Config

Recommended uploads from earlier work:
- `shared_english_corpus.csv` is recommended so the notebook can reuse the exact Q3/Q4 corpus cache immediately.
- `nnlm_en.pt` is recommended for Question 5 embedding and parameter comparisons.
- `q4_metrics_summary_en.json` is recommended for the word-level NNLM / best n-gram comparison.
- `q4_perplexity_table_en.csv` is an acceptable fallback if you do not have `q4_metrics_summary_en.json`.
- `q4_en_artifacts.zip` or `question_4_bundle.zip` can be uploaded instead of the individual Q4 English files above.
- `bpe_en.json` and `checkpoint_bpe_nnlm_en.pt` can be uploaded later if you need to resume this Q5 run after a crash.
- Nothing from Question 3 is required for Question 5.

This notebook defaults to `target_vocab_size = 5000`, because that is the only explicit BPE vocabulary-size example given in the assignment prompt. If your instructor told you to use a different size, change that one line below before training.

If only one GPU is visible, the runtime automatically falls back to that visible GPU.

In [6]:
import os
from q5_parallel_runtime import build_default_config

HF_TOKEN = os.environ.get("HF_TOKEN", "").strip()
if not HF_TOKEN:
    try:
        from kaggle_secrets import UserSecretsClient
        HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN").strip()
    except Exception:
        HF_TOKEN = ""

CONFIG = build_default_config()
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    CONFIG["hf_token"] = HF_TOKEN

CONFIG["max_pairs"] = None
CONFIG["target_vocab_size"] = 5000
CONFIG["n"] = 4
CONFIG["m"] = 100
CONFIG["h"] = 100
CONFIG["epochs"] = 30
CONFIG["batch_size"] = 512
CONFIG["lr"] = 1e-3
CONFIG["weight_decay"] = 1e-4
CONFIG["patience"] = 3
CONFIG["direct"] = False

LANG = "en"
DEVICE_ID = 0

CONFIG


{'n': 4,
 'm': 100,
 'h': 100,
 'epochs': 30,
 'batch_size': 512,
 'lr': 0.001,
 'weight_decay': 0.0001,
 'patience': 3,
 'direct': False,
 'target_vocab_size': 5000,
 'max_pairs': None,
 'hf_token': ''}

## Run Full English Pipeline

This produces the Question 5 outputs for **English**, including:
- `bpe_en.json`
- `bpe_nnlm_en.pt`
- `bpe_train_log_en.json`
- `q5_en_artifacts.zip`

If the other language's three autograder files are already present in the same session,
the runtime also creates `problem_5.zip` and `question_5.zip`.


In [7]:
from q5_parallel_runtime import run_full_language_pipeline

run_full_language_pipeline(LANG, CONFIG, device_id=DEVICE_ID)


Copied input file /kaggle/input/datasets/anirudhbankhede/q4-en-artifacts/shared_english_corpus.csv -> shared_english_corpus.csv
Copied input file /kaggle/input/datasets/anirudhbankhede/q4-en-artifacts/nnlm_en.pt -> nnlm_en.pt
Copied input file /kaggle/input/datasets/anirudhbankhede/q4-en-artifacts/q4_metrics_summary_en.json -> q4_metrics_summary_en.json
Copied input file /kaggle/input/datasets/anirudhbankhede/q4-en-artifacts/q4_perplexity_table_en.csv -> q4_perplexity_table_en.csv
Using cuda:0 -> Tesla T4
Running full Q5 pipeline for English on device_id=0
[BPE setup] English | sentences 50,000/769,121 | surface tokens 977,052 | unique surface tokens 56,224 | elapsed 00:00:05
[BPE setup] English | sentences 100,000/769,121 | surface tokens 1,945,907 | unique surface tokens 80,647 | elapsed 00:00:10
[BPE setup] English | sentences 150,000/769,121 | surface tokens 2,922,349 | unique surface tokens 99,531 | elapsed 00:00:14
[BPE setup] English | sentences 200,000/769,121 | surface tokens 

KeyError: '<0xD6>'

In [8]:
from pathlib import Path
import importlib
import q5_parallel_runtime as q5

runtime_path = Path("q5_parallel_runtime.py")
text = runtime_path.read_text(encoding="utf-8")

if "def full_base_byte_symbols():" not in text:
    text = text.replace(
        """def token_to_surface_word(token: str):
    symbol = token_to_symbol(token)
    if not symbol_terminal(symbol):
        return None
    try:
        return bytes(symbol_bytes(symbol)).decode("utf-8")
    except UnicodeDecodeError:
        return None


class ByteLevelBPE:
""",
        """def token_to_surface_word(token: str):
    symbol = token_to_symbol(token)
    if not symbol_terminal(symbol):
        return None
    try:
        return bytes(symbol_bytes(symbol)).decode("utf-8")
    except UnicodeDecodeError:
        return None


def full_base_byte_symbols():
    symbols = []
    for terminal in (False, True):
        for value in range(256):
            symbols.append(make_symbol((value,), terminal))
    return symbols


def tokenizer_has_full_byte_coverage(tokenizer) -> bool:
    required = {symbol_to_token(symbol) for symbol in full_base_byte_symbols()}
    return required.issubset(set(tokenizer.stoi))


def piece_ids_with_fallback(pieces, stoi):
    unk_id = stoi["<unk>"]
    piece_ids = []
    for piece in pieces:
        token_id = stoi.get(piece)
        if token_id is not None:
            piece_ids.append(token_id)
            continue

        symbol = token_to_symbol(piece)
        raw_bytes = symbol_bytes(symbol)
        if not raw_bytes:
            piece_ids.append(unk_id)
            continue

        for idx, value in enumerate(raw_bytes):
            terminal = idx == len(raw_bytes) - 1 and symbol_terminal(symbol)
            byte_token = symbol_to_token(make_symbol((value,), terminal))
            piece_ids.append(stoi.get(byte_token, unk_id))
    return piece_ids


class ByteLevelBPE:
"""
    )

text = text.replace(
    "        base_symbols = []\n        base_seen = set()\n",
    "        base_symbols = full_base_byte_symbols()\n        base_seen = set(base_symbols)\n",
)

old_block = """    if tokenizer_path.exists():
        tokenizer = ByteLevelBPE.load(tokenizer_path)
        saved_target = int(tokenizer.summary.get("target_vocab_size", tokenizer.target_vocab_size))
        if saved_target != int(config["target_vocab_size"]):
            raise ValueError(
                f"Existing {tokenizer_path.name} was trained with target_vocab_size={saved_target}, "
                f"but current config wants {config['target_vocab_size']}."
            )
        print(f"Loaded existing tokenizer from {tokenizer_path.name}")
        save_json(files["bpe_summary"], tokenizer.summary or {"target_vocab_size": tokenizer.target_vocab_size})
        return tokenizer
"""
new_block = """    if tokenizer_path.exists():
        tokenizer = ByteLevelBPE.load(tokenizer_path)
        saved_target = int(tokenizer.summary.get("target_vocab_size", tokenizer.target_vocab_size))
        if saved_target != int(config["target_vocab_size"]):
            raise ValueError(
                f"Existing {tokenizer_path.name} was trained with target_vocab_size={saved_target}, "
                f"but current config wants {config['target_vocab_size']}."
            )
        if not tokenizer_has_full_byte_coverage(tokenizer):
            print(
                f"Existing {tokenizer_path.name} is missing full byte coverage; retraining tokenizer.",
                flush=True,
            )
            tokenizer_path.unlink()
        else:
            print(f"Loaded existing tokenizer from {tokenizer_path.name}")
            save_json(files["bpe_summary"], tokenizer.summary or {"target_vocab_size": tokenizer.target_vocab_size})
            return tokenizer
"""
if "missing full byte coverage" not in text:
    text = text.replace(old_block, new_block)

text = text.replace(
    "            sent_ids.extend(stoi[piece] for piece in pieces)\n",
    "            sent_ids.extend(piece_ids_with_fallback(pieces, stoi))\n",
)

runtime_path.write_text(text, encoding="utf-8")

importlib.reload(q5)

LANG = globals().get("LANG", "en")
DEVICE_ID = globals().get("DEVICE_ID", 0)

q5.run_full_language_pipeline(LANG, CONFIG, device_id=DEVICE_ID)

Using cuda:0 -> Tesla T4
Running full Q5 pipeline for English on device_id=0
Existing bpe_en.json is missing full byte coverage; retraining tokenizer.
[BPE setup] English | sentences 50,000/769,121 | surface tokens 977,052 | unique surface tokens 56,224 | elapsed 00:00:05
[BPE setup] English | sentences 100,000/769,121 | surface tokens 1,945,907 | unique surface tokens 80,647 | elapsed 00:00:09
[BPE setup] English | sentences 150,000/769,121 | surface tokens 2,922,349 | unique surface tokens 99,531 | elapsed 00:00:14
[BPE setup] English | sentences 200,000/769,121 | surface tokens 3,893,977 | unique surface tokens 115,476 | elapsed 00:00:19
[BPE setup] English | sentences 250,000/769,121 | surface tokens 4,863,022 | unique surface tokens 129,573 | elapsed 00:00:23
[BPE setup] English | sentences 300,000/769,121 | surface tokens 5,834,156 | unique surface tokens 142,461 | elapsed 00:00:28
[BPE setup] English | sentences 350,000/769,121 | surface tokens 6,800,509 | unique surface tokens 